# Actividad — Búsqueda adversarial: Minimax y poda alfa-beta

## Bloque 1. Representación del juego de gato

En esta actividad implementaremos algoritmos de búsqueda adversarial utilizando el juego **Tres en línea (Tic-Tac-Toe)**.

El juego consiste en un tablero de $3\times 3$ en el que dos jugadores, `X` y `O`, colocan sus marcas alternadamente. Gana el primero que consigue colocar tres marcas consecutivas en una fila, columna o diagonal.

Consideraremos que:

- El jugador `X` corresponde a **MAX**, cuyo objetivo es maximizar la utilidad.
- El jugador `O` corresponde a **MIN**, cuyo objetivo es minimizar la utilidad.
- Ambos jugadores conocen completamente el estado del juego y sus reglas.

De acuerdo con la formulación de juegos de Russell y Norvig, necesitamos definir cuatro componentes:

1. **Estado inicial:** representa la configuración del tablero e identifica al jugador que tiene el turno.
2. **Función sucesor:** genera los movimientos legales y los estados resultantes.
3. **Test terminal:** determina si el juego ha terminado.
4. **Función de utilidad:** asigna un valor numérico a los estados terminales.

### Representación del estado

Utilizaremos un diccionario de Python con dos elementos:

- `tablero`: lista de nueve posiciones, numeradas de 0 a 8.
- `jugador`: indica a quién corresponde realizar el siguiente movimiento.

Las casillas vacías se representan mediante el carácter `' '`.

La numeración de las posiciones es:

| 0 | 1 | 2 |
|---|---|---|
| 3 | 4 | 5 |
| 6 | 7 | 8 |

El estado inicial corresponde a un tablero vacío, con `X` como primer jugador.

In [ ]:
# Estado inicial del juego
estado_inicial = {
    "tablero": [' ' for _ in range(9)],
    "jugador": 'X'
}

### Visualización del tablero

La función `mostrar_tablero()` permite visualizar la configuración de las nueve posiciones.

Recibe como argumento una lista que representa el tablero y muestra sus elementos organizados en tres filas y tres columnas.

Esta función se utilizará para observar los estados generados durante la búsqueda.

In [ ]:
def mostrar_tablero(tablero):
    print()
    for i in range(0, 9, 3):
        print(' ' + ' | '.join(tablero[i:i+3]))
        if i < 6:
            print("---+---+---")
    print()


mostrar_tablero(estado_inicial["tablero"])

### Funciones auxiliares del juego

Antes de implementar los componentes de la formulación de juegos, definiremos tres funciones auxiliares:

- `movimientos_validos(tablero)`: devuelve una lista con los índices de las casillas vacías.
- `ganador(tablero, jugador)`: determina si un jugador ha completado tres marcas consecutivas.
- `tablero_lleno(tablero)`: comprueba si todas las casillas están ocupadas.

Estas funciones implementan reglas básicas del juego y serán utilizadas posteriormente por la función sucesor y el test terminal.

In [ ]:
def movimientos_validos(tablero):
    return [i for i in range(9) if tablero[i] == ' ']

In [ ]:
def ganador(tablero, jugador):
    combinaciones = [
        (0,1,2), (3,4,5), (6,7,8),  # Filas
        (0,3,6), (1,4,7), (2,5,8),  # Columnas
        (0,4,8), (2,4,6)            # Diagonales
    ]

    return any(all(tablero[i] == jugador for i in c)
               for c in combinaciones)

In [ ]:
def tablero_lleno(tablero):
    return all(c != ' ' for c in tablero)

## Bloque 2. Formulación del juego

### Test terminal

El **test terminal** determina si un estado corresponde a una partida que ha finalizado.

En el juego de gato, una partida termina cuando se cumple alguna de las siguientes condiciones:

- El jugador `X` consigue tres marcas consecutivas.
- El jugador `O` consigue tres marcas consecutivas.
- Todas las casillas están ocupadas y ningún jugador ha ganado.

La función `test_terminal(estado)` devuelve `True` cuando se cumple alguna de estas condiciones y `False` en caso contrario.

Observa que la función recibe el **estado completo**, aunque solamente necesita consultar la configuración del tablero.

In [ ]:
def test_terminal(estado):
    tablero = estado["tablero"]

    return (ganador(tablero, 'X') or
            ganador(tablero, 'O') or
            tablero_lleno(tablero))

### Función de utilidad

La **función de utilidad** asigna un valor numérico a los estados terminales, desde la perspectiva del jugador MAX (`X`).

Utilizaremos la siguiente definición:

$$
U(s)=
\begin{cases}
+1 & \text{si gana X (MAX)}\\
-1 & \text{si gana O (MIN)}\\
0 & \text{si hay empate}
\end{cases}
$$

La función `utilidad(estado)` implementa esta definición.

Es importante distinguir entre el **test terminal**, que determina si el juego ha finalizado, y la **función de utilidad**, que establece el resultado de la partida.

En los algoritmos Minimax y poda alfa-beta, la función de utilidad se utilizará para evaluar los estados terminales.

In [ ]:
def utilidad(estado):
    tablero = estado["tablero"]

    if ganador(tablero, 'X'):
        return 1
    elif ganador(tablero, 'O'):
        return -1
    else:
        return 0

### Función sucesor

La **función sucesor** genera los movimientos legales que pueden realizarse desde un estado y los estados que resultan de aplicar cada movimiento.

Formalmente:

$$
SUCESORES(s)=\{(a_1,s_1),(a_2,s_2),\ldots,(a_n,s_n)\}
$$

donde:

- $a_i$ representa una acción legal.
- $s_i$ representa el estado resultante de ejecutar la acción $a_i$.

Para implementar esta función debemos considerar que:

1. Si el estado es terminal, no existen sucesores.
2. Cada acción consiste en colocar la marca del jugador actual en una casilla vacía.
3. La ejecución de una acción produce un nuevo tablero.
4. Después de realizar un movimiento, el turno corresponde al otro jugador.

La función `sucesores(estado)` devolverá una lista de pares `(accion, nuevo_estado)`.

**Importante:** para generar cada nuevo estado utilizaremos `tablero.copy()`. Esto evita modificar el tablero original durante la exploración del árbol de búsqueda.

In [ ]:
def sucesores(estado):
    tablero = estado["tablero"]
    jugador = estado["jugador"]

    if test_terminal(estado):
        return []

    lista = []

    for accion in movimientos_validos(tablero):

        nuevo_tablero = tablero.copy()
        nuevo_tablero[accion] = jugador

        siguiente_jugador = 'O' if jugador == 'X' else 'X'

        nuevo_estado = {
            "tablero": nuevo_tablero,
            "jugador": siguiente_jugador
        }

        lista.append((accion, nuevo_estado))

    return lista

### Exploración de los estados sucesores

Para comprobar el funcionamiento de la función sucesor, utilizaremos el estado inicial del juego.

Como el tablero se encuentra vacío, el jugador `X` dispone de nueve movimientos legales.

Ejecutaremos `sucesores(estado_inicial)` y mostraremos los estados resultantes para observar cómo se modifica el tablero y cómo cambia el turno.

Cada estado generado representa una posible continuación del juego.

In [ ]:
for accion, estado in sucesores(estado_inicial):
    print("Acción:", accion)
    print("Siguiente jugador:", estado["jugador"])
    mostrar_tablero(estado["tablero"])

### Comprobación del test terminal y la función de utilidad

Consideremos tres estados que representan diferentes resultados del juego:

- Una victoria del jugador `X`.
- Una victoria del jugador `O`.
- Un empate.

Para cada estado, comprobaremos si la partida ha terminado y calcularemos su utilidad.

Observa que todos estos estados deben satisfacer el test terminal, pero sus valores de utilidad son diferentes.

In [ ]:
estados = [
    {
        "tablero": ['X','X','X',
                    'O','O',' ',
                    ' ',' ',' '],
        "jugador": 'O'
    },
    {
        "tablero": ['O','O','O',
                    'X','X',' ',
                    'X',' ',' '],
        "jugador": 'X'
    },
    {
        "tablero": ['X','O','X',
                    'X','O','O',
                    'O','X','X'],
        "jugador": 'O'
    }
]

for estado in estados:
    mostrar_tablero(estado["tablero"])
    print("¿Estado terminal?", test_terminal(estado))
    print("Utilidad:", utilidad(estado))
    print()

## Bloque 3. Algoritmo Minimax

El algoritmo **Minimax** es una estrategia de búsqueda adversarial utilizada en juegos de dos jugadores, deterministas, de suma cero e información perfecta.

Su objetivo es seleccionar la mejor acción posible, suponiendo que ambos jugadores toman decisiones óptimas.

En nuestro juego de gato:

- **MAX (`X`)** busca maximizar el valor de utilidad.
- **MIN (`O`)** busca minimizar el valor de utilidad.

El algoritmo explora recursivamente el árbol del juego hasta alcanzar estados terminales, donde aplica la función `utilidad(estado)`.

Posteriormente, los valores obtenidos se propagan hacia los niveles superiores del árbol:

- En los nodos MAX se selecciona el **mayor valor** de sus sucesores.
- En los nodos MIN se selecciona el **menor valor** de sus sucesores.

De esta manera, el valor Minimax de un estado representa el resultado que puede garantizar el jugador MAX, suponiendo que ambos jugadores actúan óptimamente.

### Funciones MAX_VALOR y MIN_VALOR

La implementación de Minimax utiliza dos funciones recursivas que se llaman mutuamente.

**Función `max_valor(estado)`**

Calcula el valor Minimax de un estado en el que corresponde jugar a MAX.

1. Si el estado es terminal, devuelve su utilidad.
2. Inicializa el valor $v=-\infty$.
3. Genera los estados sucesores.
4. Para cada sucesor, calcula el valor que obtendría MIN.
5. Devuelve el mayor de los valores obtenidos.

**Función `min_valor(estado)`**

Calcula el valor Minimax de un estado en el que corresponde jugar a MIN.

1. Si el estado es terminal, devuelve su utilidad.
2. Inicializa el valor $v=+\infty$.
3. Genera los estados sucesores.
4. Para cada sucesor, calcula el valor que obtendría MAX.
5. Devuelve el menor de los valores obtenidos.

Estas funciones implementan la siguiente relación recursiva:

$$
V(s)=
\begin{cases}
U(s), & \text{si }s\text{ es terminal}\\
\max\limits_{s'\in S(s)}V(s'), & \text{si juega MAX}\\
\min\limits_{s'\in S(s)}V(s'), & \text{si juega MIN}
\end{cases}
$$

donde $S(s)$ representa el conjunto de estados sucesores de $s$.

In [ ]:
def max_valor(estado):
    if test_terminal(estado):
        return utilidad(estado)

    v = -float('inf')

    for accion, s in sucesores(estado):
        v = max(v, min_valor(s))

    return v

In [ ]:
def min_valor(estado):
    if test_terminal(estado):
        return utilidad(estado)

    v = float('inf')

    for accion, s in sucesores(estado):
        v = min(v, max_valor(s))

    return v

### Función DECISIÓN-MINIMAX

Las funciones `max_valor()` y `min_valor()` calculan valores de utilidad, pero no indican directamente qué movimiento debe realizar el jugador.

Para seleccionar una acción utilizaremos la función `decision_minimax(estado)`.

Esta función considera que el turno corresponde a MAX (`X`) y realiza las siguientes operaciones:

1. Genera los sucesores del estado actual.
2. Evalúa cada sucesor mediante `min_valor()`, porque después de que juega MAX corresponde el turno de MIN.
3. Conserva la acción que produce el mayor valor.
4. Devuelve la acción seleccionada.

Si varias acciones tienen el mismo valor máximo, la implementación conserva la primera que encuentra.

**Importante:** `decision_minimax()` devuelve una acción (índice del tablero), mientras que `max_valor()` y `min_valor()` devuelven valores de utilidad.

In [ ]:
def decision_minimax(estado):
    mejor_valor = -float('inf')
    mejor_accion = None

    for accion, s in sucesores(estado):
        v = min_valor(s)

        if v > mejor_valor:
            mejor_valor = v
            mejor_accion = accion

    return mejor_accion

### Ejemplo: selección de una acción mediante Minimax

Consideremos un estado intermedio del juego en el que corresponde jugar a MAX.

El jugador `X` tiene dos marcas consecutivas en la primera fila, por lo que puede ganar colocando una tercera marca en la posición 2.

Utilizaremos `decision_minimax()` para determinar qué acción selecciona el algoritmo.

In [ ]:
estado = {
    "tablero": [
        'X', 'X', ' ',
        'O', 'O', ' ',
        ' ', ' ', ' '
    ],
    "jugador": 'X'
}

mostrar_tablero(estado["tablero"])

accion = decision_minimax(estado)

print("Acción seleccionada:", accion)

### Actividad 1. Evaluación de las posiciones iniciales

Consideremos el estado inicial del juego de gato, en el que todas las casillas están vacías y corresponde jugar a MAX (`X`).

**Objetivo:** determinar el valor Minimax de cada una de las nueve posibles acciones iniciales y explicar su significado.

Realiza lo siguiente:

1. Utiliza la función `sucesores()` para generar los nueve estados que resultan de la primera jugada de `X`.
2. Para cada estado sucesor, calcula su valor Minimax mediante la función `min_valor()`.
3. Muestra el índice de cada acción y el valor obtenido.
4. Ejecuta `decision_minimax()` sobre el tablero vacío e identifica la acción seleccionada.

**Responde en una celda Markdown:**

- ¿Qué valor Minimax obtuviste para cada una de las nueve acciones?
- ¿Por qué jugar en el centro o en una esquina produce el mismo valor Minimax?
- ¿Qué significa obtener un valor Minimax igual a cero?
- ¿Por qué `decision_minimax()` selecciona una acción específica si varias acciones tienen el mismo valor?

Justifica tus respuestas considerando que ambos jugadores toman decisiones óptimas.

In [ ]:
# Estado inicial
estado = {
    "tablero": [' '] * 9,
    "jugador": 'X'
}

# Evaluar las acciones iniciales
for accion, s in sucesores(estado):

    # TODO: Calcular el valor Minimax del sucesor
    valor = ...

    print(f"Acción: {accion}, Valor Minimax: {valor}")


# TODO: Obtener la acción seleccionada por Minimax
mejor_accion = ...

print("Acción seleccionada:", mejor_accion)

## Bloque 4. Poda alfa-beta

El algoritmo **poda alfa-beta** es una optimización de Minimax que permite evitar la exploración de ramas del árbol de juego que no pueden modificar la decisión óptima.

Para ello, utiliza dos parámetros:

- **Alfa ($\alpha$):** mejor valor que MAX puede garantizar hasta el momento en el camino de búsqueda.
- **Beta ($\beta$):** mejor valor que MIN puede garantizar hasta el momento en el camino de búsqueda.

Durante la exploración, estos valores se actualizan conforme se evalúan los estados sucesores.

La condición para realizar una poda es:

$$
\alpha \geq \beta
$$

Cuando se cumple esta condición, podemos interrumpir la exploración de los sucesores restantes del nodo, porque no pueden modificar la decisión de sus antecesores.

La poda alfa-beta conserva el resultado de Minimax, pero puede reducir considerablemente el número de estados explorados. Su eficiencia depende, entre otros factores, del orden en que se examinan los movimientos.

### Actividad 2. Implementación de poda alfa-beta

Modifica las funciones `max_valor()` y `min_valor()` para incorporar los parámetros $\alpha$ y $\beta$.

Para ello:

1. Agrega los parámetros `alfa` y `beta` a las funciones recursivas.
2. Actualiza `alfa` cuando MAX encuentre un valor mayor.
3. Actualiza `beta` cuando MIN encuentre un valor menor.
4. Interrumpe la exploración cuando se cumpla la condición de poda.
5. Conserva la evaluación de los estados terminales mediante `utilidad(estado)`.

Utiliza como referencia los pseudocódigos explicados en clase.

In [ ]:

def max_valor_ab(estado, alfa, beta):
    if test_terminal(estado):
        return utilidad(estado)

    v = -float('inf')

    for accion, s in sucesores(estado):

        # TODO: Calcular el valor del sucesor
        v = ...

        # TODO: Comprobar la condición de poda
        if ...:
            return v

        # TODO: Actualizar alfa
        alfa = ...

    return v

In [ ]:
def min_valor_ab(estado, alfa, beta):
    if test_terminal(estado):
        return utilidad(estado)

    v = float('inf')

    for accion, s in sucesores(estado):

        # TODO: Calcular el valor del sucesor
        v = ...

        # TODO: Comprobar la condición de poda
        if ...:
            return v

        # TODO: Actualizar beta
        beta = ...

    return v

### Función de decisión con poda alfa-beta

Al igual que `decision_minimax()`, necesitamos una función que seleccione la mejor acción para MAX.

La función `decision_alfa_beta(estado)` deberá:

1. Inicializar $\alpha=-\infty$ y $\beta=+\infty$.
2. Generar los sucesores del estado.
3. Evaluar cada sucesor mediante `min_valor_ab()`.
4. Conservar la acción que produzca el mayor valor.
5. Actualizar $\alpha$ después de evaluar cada movimiento.

La función devolverá la acción seleccionada.

Recuerda que, después de ejecutar una acción de MAX, corresponde jugar a MIN.

In [ ]:
def decision_alfa_beta(estado):
    mejor_valor = -float('inf')
    mejor_accion = None

    alfa = -float('inf')
    beta = float('inf')

    for accion, s in sucesores(estado):

        # TODO: Evaluar el sucesor utilizando poda alfa-beta
        v = ...

        if v > mejor_valor:
            mejor_valor = v
            mejor_accion = accion

        # TODO: Actualizar alfa
        alfa = ...

    return mejor_accion

### Verificación de la poda alfa-beta

Una implementación correcta de poda alfa-beta debe producir el mismo valor de utilidad que Minimax.

Considera el siguiente estado, en el que corresponde jugar a MAX:

```text
 X | X |  
---+---+---
 O | O |  
---+---+---
   |   |  
```

Ejecuta las funciones `decision_minimax()` y `decision_alfa_beta()` y compara sus resultados.

Posteriormente, repite la comprobación utilizando el tablero vacío.

**Responde:**

1. ¿Ambos algoritmos seleccionan la misma acción?
2. ¿La poda alfa-beta modifica el valor Minimax de un estado?
3. ¿Cuál es la ventaja de utilizar poda alfa-beta si ambos algoritmos producen el mismo resultado?

In [ ]:
estado = {
    "tablero": [
        'X', 'X', ' ',
        'O', 'O', ' ',
        ' ', ' ', ' '
    ],
    "jugador": 'X'
}

mostrar_tablero(estado["tablero"])

print("Minimax:", decision_minimax(estado))
print("Alfa-beta:", decision_alfa_beta(estado))

In [ ]:
estado = {
    "tablero": [' '] * 9,
    "jugador": 'X'
}

valor_minimax = max_valor(estado)

valor_alfa_beta = max_valor_ab(
    estado,
    -float('inf'),
    float('inf')
)

print("Valor Minimax:", valor_minimax)
print("Valor alfa-beta:", valor_alfa_beta)

assert valor_minimax == valor_alfa_beta

## Bloque 5. Comparación de Minimax y poda alfa-beta

Hasta ahora hemos implementado dos algoritmos de búsqueda adversarial:

- **Minimax**, que explora recursivamente el árbol del juego hasta alcanzar los estados terminales.
- **Poda alfa-beta**, que evita explorar ramas que no pueden modificar la decisión óptima.

Ambos algoritmos calculan el mismo valor Minimax, pero pueden diferir considerablemente en el número de estados que necesitan explorar.

Para comparar su eficiencia, utilizaremos el **número de estados expandidos** como medida del esfuerzo computacional.

Consideraremos que un estado es expandido cuando el algoritmo genera sus sucesores. Por lo tanto, incrementaremos un contador cada vez que se expanda un estado no terminal.

Los estados terminales no se contabilizarán como expandidos, ya que únicamente se evalúan mediante la función de utilidad.

### Actividad 3. Medición de estados expandidos

Modifica las funciones de Minimax y poda alfa-beta para contabilizar el número de expansiones realizadas durante la búsqueda.

Para ello:

1. Incorpora un contador de expansiones en cada algoritmo.
2. Incrementa el contador cuando se expanda un estado no terminal.
3. Conserva las reglas de evaluación y selección de acciones.
4. Ejecuta ambos algoritmos sobre el mismo estado inicial.
5. Compara los valores Minimax y el número de estados expandidos.

**Importante:** utiliza el mismo orden de generación de sucesores para ambos algoritmos.

### Contador de expansiones

Utilizaremos un diccionario con la siguiente estructura:

```python
contador = {"expansiones": 0}
```

El contador se pasará como argumento a las funciones recursivas.

Cada vez que se expanda un estado no terminal, se incrementará su valor:

```python
contador["expansiones"] += 1
```

Al finalizar la búsqueda, el diccionario contendrá el número total de estados expandidos.

Esta modificación permitirá medir el trabajo realizado por ambos algoritmos sin alterar su lógica de búsqueda.

In [ ]:
def max_valor_cont(estado, contador):
    if test_terminal(estado):
        return utilidad(estado)

    # TODO: Incrementar el contador de expansiones

    v = -float('inf')

    for accion, s in sucesores(estado):
        v = max(v, min_valor_cont(s, contador))

    return v

In [ ]:
def min_valor_cont(estado, contador):
    if test_terminal(estado):
        return utilidad(estado)

    # TODO: Incrementar el contador de expansiones

    v = float('inf')

    for accion, s in sucesores(estado):
        v = min(v, max_valor_cont(s, contador))

    return v

In [ ]:
def max_valor_ab_cont(estado, alfa, beta, contador):
    if test_terminal(estado):
        return utilidad(estado)

    # TODO: Incrementar el contador de expansiones

    v = -float('inf')

    for accion, s in sucesores(estado):
        v = max(v, min_valor_ab_cont(s, alfa, beta, contador))

        if v >= beta:
            return v

        alfa = max(alfa, v)

    return v

In [ ]:
def min_valor_ab_cont(estado, alfa, beta, contador):
    if test_terminal(estado):
        return utilidad(estado)

    # TODO: Incrementar el contador de expansiones

    v = float('inf')

    for accion, s in sucesores(estado):
        v = min(v, max_valor_ab_cont(s, alfa, beta, contador))

        if v <= alfa:
            return v

        beta = min(beta, v)

    return v

### Experimento: búsqueda desde el tablero vacío

Evaluaremos el estado inicial del juego utilizando Minimax y poda alfa-beta.

Ambos algoritmos deberán calcular el mismo valor, pero esperamos que poda alfa-beta expanda menos estados.

Para realizar la comparación:

1. Inicializa un contador independiente para cada algoritmo.
2. Ejecuta `max_valor_cont()` y `max_valor_ab_cont()` sobre el tablero vacío.
3. Registra el valor Minimax y el número de expansiones de cada algoritmo.
4. Calcula el porcentaje de reducción de expansiones.

El porcentaje de reducción se define como:

$$
R=\frac{N_{\mathrm{Minimax}}-N_{\alpha\beta}}
{N_{\mathrm{Minimax}}}\times100
$$

donde:

- $N_{\mathrm{Minimax}}$ es el número de estados expandidos por Minimax.
- $N_{\alpha\beta}$ es el número de estados expandidos por poda alfa-beta.

In [ ]:
estado = {
    "tablero": [' '] * 9,
    "jugador": 'X'
}

contador_minimax = {"expansiones": 0}
contador_ab = {"expansiones": 0}

# TODO: Ejecutar Minimax con contador
valor_minimax = ...

# TODO: Ejecutar alfa-beta con contador
valor_ab = ...

n_minimax = contador_minimax["expansiones"]
n_ab = contador_ab["expansiones"]

# TODO: Calcular el porcentaje de reducción
reduccion = ...

print("Minimax")
print("Valor:", valor_minimax)
print("Estados expandidos:", n_minimax)

print("\nPoda alfa-beta")
print("Valor:", valor_ab)
print("Estados expandidos:", n_ab)

print(f"\nReducción de expansiones: {reduccion:.2f}%")

### Análisis de resultados

A partir de los resultados obtenidos, responde en una celda Markdown:

1. ¿Ambos algoritmos calcularon el mismo valor Minimax para el tablero vacío?
2. ¿Cuántos estados expandió cada algoritmo?
3. ¿Qué porcentaje de reducción se obtuvo mediante poda alfa-beta?
4. ¿Por qué alfa-beta puede evitar la exploración de ciertos estados sin modificar el resultado?
5. ¿El número de estados expandidos por alfa-beta depende del orden en que se generan los sucesores? Justifica tu respuesta.

**Conclusión:** explica brevemente qué ventaja ofrece la poda alfa-beta respecto de Minimax y por qué esta optimización resulta importante cuando aumenta el tamaño del árbol de juego.

## Bloque 6. Reto: Minimax con profundidad limitada

Hasta ahora hemos utilizado Minimax y poda alfa-beta para explorar el árbol del juego hasta alcanzar estados terminales. Sin embargo, cuando el espacio de búsqueda es demasiado grande, explorar todas las posibles continuaciones puede resultar computacionalmente costoso.

Una alternativa consiste en **limitar la profundidad de búsqueda** y utilizar una función de evaluación heurística para estimar el valor de los estados que todavía no son terminales.

Para ello, modificaremos Minimax incorporando dos elementos:

1. **Límite de profundidad:** determina el número máximo de movimientos que podemos explorar.
2. **Función de evaluación heurística (`eval_estado`)**: estima qué tan favorable es un estado para MAX cuando la búsqueda alcanza el límite establecido.

La evaluación de un estado se realizará considerando:

$$
V(s,d)=
\begin{cases}
U(s), & \text{si }s\text{ es terminal}\\
EVAL(s), & \text{si }d=0\\
\max V(s',d-1), & \text{si juega MAX}\\
\min V(s',d-1), & \text{si juega MIN}
\end{cases}
$$

donde $d$ representa el número de movimientos que todavía pueden explorarse.

En este reto utilizaremos una **profundidad máxima de cuatro movimientos**, considerando tanto las acciones de MAX como las de MIN.

### Función de evaluación heurística

Para estimar el valor de un estado no terminal, consideraremos las líneas del tablero que todavía pueden ser completadas por los jugadores.

Una línea corresponde a cualquiera de las ocho combinaciones ganadoras: tres filas, tres columnas y dos diagonales.

Definimos las siguientes características:

- $L_{2X}(s)$: número de líneas con dos marcas `X` y una casilla vacía.
- $L_{2O}(s)$: número de líneas con dos marcas `O` y una casilla vacía.
- $L_{1X}(s)$: número de líneas con una marca `X` y dos casillas vacías.
- $L_{1O}(s)$: número de líneas con una marca `O` y dos casillas vacías.

La función heurística será:

$$
EVAL(s)=10[L_{2X}(s)-L_{2O}(s)]
+[L_{1X}(s)-L_{1O}(s)]
$$

Esta función asigna mayor importancia a las líneas que están próximas a completarse.

- Un valor positivo indica una posición favorable para MAX.
- Un valor negativo indica una posición favorable para MIN.
- Un valor cercano a cero indica que las características consideradas están relativamente equilibradas.

**Importante:** el resultado de esta función es una estimación heurística, no el valor Minimax exacto del estado.

Para los estados terminales utilizaremos valores de $+100$, $-100$ y $0$, de manera que una victoria real tenga preferencia sobre cualquier puntuación heurística de un estado no terminal.

In [ ]:
def eval_estado(estado):
    tablero = estado["tablero"]

    # Evaluación de estados terminales
    if test_terminal(estado):
        return 100 * utilidad(estado)

    combinaciones = [
        (0,1,2), (3,4,5), (6,7,8),
        (0,3,6), (1,4,7), (2,5,8),
        (0,4,8), (2,4,6)
    ]

    valor = 0

    for combinacion in combinaciones:
        linea = [tablero[i] for i in combinacion]

        x = linea.count('X')
        o = linea.count('O')

        if x == 2 and o == 0:
            valor += 10
        elif o == 2 and x == 0:
            valor -= 10
        elif x == 1 and o == 0:
            valor += 1
        elif o == 1 and x == 0:
            valor -= 1

    return valor

### Reto. Implementación de Minimax con profundidad limitada

Modifica las funciones `max_valor()` y `min_valor()` para incorporar un parámetro adicional denominado `profundidad`.

Este parámetro indica cuántos movimientos adicionales puede explorar el algoritmo.

Para completar la implementación:

1. Conserva el test terminal. Cuando el juego haya terminado, devuelve la evaluación del estado terminal.
2. Si la profundidad es cero, devuelve la estimación calculada mediante `eval_estado()`.
3. Conserva la selección del máximo y mínimo valor según corresponda.
4. Disminuye la profundidad en una unidad cada vez que explores un estado sucesor.
5. Modifica la función de decisión para ejecutar la búsqueda con una profundidad máxima de cuatro movimientos.

**Recuerda:** la primera acción que se evalúa en la función de decisión también cuenta como uno de los cuatro movimientos permitidos.

In [ ]:
def max_valor_lim(estado, profundidad):
    if test_terminal(estado):
        return eval_estado(estado)

    # TODO: Comprobar el límite de profundidad
    if ...:
        return ...

    v = -float('inf')

    for accion, s in sucesores(estado):

        # TODO: Evaluar el sucesor reduciendo la profundidad
        v = ...

    return v

In [ ]:
def min_valor_lim(estado, profundidad):
    if test_terminal(estado):
        return eval_estado(estado)

    # TODO: Comprobar el límite de profundidad
    if ...:
        return ...

    v = float('inf')

    for accion, s in sucesores(estado):

        # TODO: Evaluar el sucesor reduciendo la profundidad
        v = ...

    return v

In [ ]:
def decision_minimax_lim(estado, profundidad):
    mejor_valor = -float('inf')
    mejor_accion = None

    for accion, s in sucesores(estado):

        # TODO: Evaluar el sucesor considerando
        # que ya se realizó un movimiento
        v = ...

        if v > mejor_valor:
            mejor_valor = v
            mejor_accion = accion

    return mejor_accion

### Comparación de Minimax completo y Minimax limitado

Utiliza el tablero vacío para comparar las acciones seleccionadas por ambos algoritmos.

1. Ejecuta `decision_minimax()` para obtener la acción seleccionada por Minimax completo.
2. Ejecuta `decision_minimax_lim()` utilizando una profundidad máxima de cuatro movimientos.
3. Compara las acciones seleccionadas.
4. Repite el experimento utilizando profundidades máximas de 2, 3 y 4 movimientos.

Registra tus resultados y responde:

- ¿Minimax completo y Minimax limitado seleccionan necesariamente la misma acción?
- ¿Cómo influye la profundidad de búsqueda en la decisión seleccionada?
- ¿Por qué una función de evaluación heurística puede producir una decisión diferente de la obtenida mediante Minimax completo?
- ¿Qué ventaja computacional tiene limitar la profundidad de búsqueda?

In [ ]:
estado = {
    "tablero": [' '] * 9,
    "jugador": 'X'
}

print("Minimax completo:")
print("Acción:", decision_minimax(estado))

print("\nMinimax con profundidad limitada:")

for profundidad in [2, 3, 4]:

    # TODO: Obtener la acción seleccionada
    accion = ...

    print(f"Profundidad {profundidad}: Acción {accion}")

### Reflexión final

A lo largo de esta actividad implementamos tres aproximaciones para resolver problemas de búsqueda adversarial:

- **Minimax:** determina el resultado óptimo explorando el árbol hasta los estados terminales.
- **Poda alfa-beta:** evita explorar ramas que no pueden modificar el resultado Minimax.
- **Minimax con profundidad limitada:** reduce la exploración mediante un límite de profundidad y una función de evaluación heurística.

Las tres aproximaciones muestran diferentes estrategias para enfrentar el costo computacional de la búsqueda adversarial.

**Pregunta final:**

¿Por qué limitar la profundidad de búsqueda puede resultar necesario en juegos más complejos, como el ajedrez o Go, incluso cuando disponemos de poda alfa-beta?

Explica tu respuesta considerando el tamaño del árbol de búsqueda, el costo computacional y la calidad de las decisiones.